# 실습0) 토큰화와 임베딩 이해하기

RAG 시스템의 첫 단계는 텍스트를 컴퓨터가 이해할 수 있는 숫자로 바꾸는 것입니다. 이 실습에서는 그 과정을 직접 눈으로 확인해봅니다.

1. **토큰화(Tokenization)**: 문장을 어떻게 잘게 쪼개는지
2. **단어 사전(Vocabulary)**: 모델이 알고 있는 단어들은 무엇인지
3. **임베딩(Embedding)**: 토큰이 어떻게 숫자 벡터로 바뀌는지
4. **문장 유사도**: 두 문장의 임베딩 벡터로 의미가 얼마나 비슷한지 계산

한국어 소형 BERT 모델인 `klue/bert-base`를 사용합니다.

## 1. 모델과 토크나이저 불러오기

`klue/bert-base`는 한국어 텍스트로 학습된 소형 BERT 모델입니다. `AutoTokenizer`는 텍스트를 토큰으로 쪼개는 역할을, `AutoModel`은 토큰을 임베딩 벡터로 변환하는 역할을 합니다.

In [1]:
import torch
from transformers import AutoTokenizer, AutoModel

# klue/bert-base: 한국어로 사전 학습된 소형 BERT 모델
# 처음 실행 시 허깅페이스 허브에서 모델 파일을 자동으로 다운로드합니다 (약 400MB)
MODEL_NAME = "klue/bert-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
model.eval()

print(f"모델: {MODEL_NAME}")
print(f"단어 사전 크기: {tokenizer.vocab_size:,}개")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: klue/bert-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


모델: klue/bert-base
단어 사전 크기: 32,000개


## 2. 토큰화(Tokenization) 결과 확인하기

문장을 토크나이저에 넣으면 의미 단위(주로 형태소나 음절 조각)로 쪼개집니다. BERT는 단어를 통째로 사전에 담기보다, 자주 등장하는 조각(subword)으로 쪼개서 처리합니다. `##`이 붙은 토큰은 앞 토큰에 이어 붙는 조각이라는 뜻입니다.

In [2]:
sentence = "검색증강생성 기법으로 챗봇의 답변 신뢰도를 높일 수 있다."

# tokenize(): 문장을 서브워드 토큰 리스트로 분리
tokens = tokenizer.tokenize(sentence)

# encode(): 각 토큰을 사전에 정의된 고유 숫자(ID)로 변환
token_ids = tokenizer.encode(sentence)

print(f"원문: {sentence}\n")
print(f"토큰 개수: {len(tokens)}개")
print(f"토큰 목록: {tokens}\n")
print(f"토큰 ID(CLS/SEP 포함): {token_ids}")

원문: 검색증강생성 기법으로 챗봇의 답변 신뢰도를 높일 수 있다.

토큰 개수: 16개
토큰 목록: ['검색', '##증', '##강', '##생', '##성', '기법', '##으로', '[UNK]', '답변', '신뢰도', '##를', '높일', '수', '있', '##다', '.']

토큰 ID(CLS/SEP 포함): [2, 6102, 2304, 2280, 2065, 2047, 7385, 6233, 1, 6272, 14605, 2138, 9213, 1295, 1513, 2062, 18, 3]


## 3. 단어 사전(Vocabulary) 살펴보기

토크나이저는 내부적으로 "토큰 → 고유 ID" 매핑 테이블(단어 사전)을 가지고 있습니다. 이 사전에 없는 단어는 여러 조각으로 쪼개서라도 표현합니다. 사전의 일부와, 특정 단어의 ID를 직접 찾아봅니다.

In [3]:
vocab = tokenizer.get_vocab()  # {토큰 문자열: ID} 형태의 딕셔너리

print(f"전체 단어 사전 크기: {len(vocab):,}개\n")

# 사전 일부를 무작위로 살펴보기 (ID 5000~5010번)
sample_items = sorted(vocab.items(), key=lambda x: x[1])[5000:5010]
print("사전 샘플 (ID 5000~5010):")
for token, idx in sample_items:
    print(f"  {idx}: {token}")

# 특정 단어가 사전에 있는지, 있다면 ID가 무엇인지 확인
for word in ["챗봇", "임베딩", "RAG"]:
    token_id = tokenizer.convert_tokens_to_ids(word)
    is_known = word in vocab
    print(f"\n'{word}' → 사전에 있음: {is_known}, ID: {token_id}")

전체 단어 사전 크기: 32,000개

사전 샘플 (ID 5000~5010):
  5000: 등급
  5001: 그리스
  5002: 36
  5003: 고등
  5004: 온라인
  5005: 예방
  5006: 본인
  5007: 혜택
  5008: 해양
  5009: 스마트폰

'챗봇' → 사전에 있음: False, ID: 1

'임베딩' → 사전에 있음: False, ID: 1

'RAG' → 사전에 있음: False, ID: 1


## 4. 임베딩(Embedding) 벡터 확인하기

토큰 ID를 BERT 모델에 통과시키면, 각 토큰마다 의미 정보를 담은 실수 벡터(임베딩)가 나옵니다. 문장 전체를 하나의 벡터로 표현하고 싶을 때는 보통 모든 토큰 벡터의 평균(mean pooling)을 사용합니다.

In [4]:
def embed_sentence(text: str) -> torch.Tensor:
    """문장을 입력받아 BERT의 mean pooling 문장 임베딩 벡터를 반환한다."""
    inputs = tokenizer(text, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)

    # outputs.last_hidden_state: (배치=1, 토큰 수, 768차원) 형태의 토큰별 임베딩
    token_embeddings = outputs.last_hidden_state[0]
    sentence_embedding = token_embeddings.mean(dim=0)  # 토큰 벡터들을 평균내어 문장 벡터 하나로
    return sentence_embedding


sample_vector = embed_sentence(sentence)
print(f"문장: {sentence}")
print(f"임베딩 벡터 차원: {sample_vector.shape}")
print(f"벡터 앞부분 5개 값: {sample_vector[:5].tolist()}")

문장: 검색증강생성 기법으로 챗봇의 답변 신뢰도를 높일 수 있다.
임베딩 벡터 차원: torch.Size([768])
벡터 앞부분 5개 값: [-0.1497098207473755, 0.04980951547622681, 0.27129825949668884, 1.3690714836120605, 0.519431471824646]


## 5. 문장 유사도 계산하기

두 문장의 임베딩 벡터가 있으면, 코사인 유사도(cosine similarity)로 의미가 얼마나 비슷한지 수치로 비교할 수 있습니다. 값은 -1~1 사이이며, 1에 가까울수록 의미가 유사합니다. 의미가 비슷한 문장 쌍과, 전혀 다른 문장 쌍을 함께 비교해봅니다.

In [5]:
def cosine_similarity(vec1: torch.Tensor, vec2: torch.Tensor) -> float:
    """두 벡터 사이의 코사인 유사도를 계산한다 (1에 가까울수록 유사)."""
    return torch.nn.functional.cosine_similarity(vec1.unsqueeze(0), vec2.unsqueeze(0)).item()


sentence_pairs = [
    ("오늘 날씨가 정말 좋다.", "날씨가 참 화창하네요."),          # 의미가 비슷한 문장
    ("고양이가 소파 위에서 자고 있다.", "강아지가 마당에서 뛰어논다."),  # 주제는 비슷하지만 다른 문장
    ("오늘 날씨가 정말 좋다.", "주가지수가 큰 폭으로 하락했다."),   # 전혀 관련 없는 문장
]

for sent_a, sent_b in sentence_pairs:
    vec_a = embed_sentence(sent_a)
    vec_b = embed_sentence(sent_b)
    similarity = cosine_similarity(vec_a, vec_b)
    print(f"A: {sent_a}")
    print(f"B: {sent_b}")
    print(f"유사도: {similarity:.4f}\n")

A: 오늘 날씨가 정말 좋다.
B: 날씨가 참 화창하네요.
유사도: 0.8100

A: 고양이가 소파 위에서 자고 있다.
B: 강아지가 마당에서 뛰어논다.
유사도: 0.8133

A: 오늘 날씨가 정말 좋다.
B: 주가지수가 큰 폭으로 하락했다.
유사도: 0.5600

